<a href="https://colab.research.google.com/github/musowjanya/Natural-Language-Processing/blob/main/Simple_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
corpus = [
    "I love machine learning",
    "wordvec is great algorithm",
    "Implementing word2vec is fun"
]

In [4]:
sentences = [s.lower().split() for s in corpus]

In [5]:
sentences

[['i', 'love', 'machine', 'learning'],
 ['wordvec', 'is', 'great', 'algorithm'],
 ['implementing', 'word2vec', 'is', 'fun']]

In [6]:
vocab = sorted(set(word for sent in sentences for word in sent))

In [7]:
vocab

['algorithm',
 'fun',
 'great',
 'i',
 'implementing',
 'is',
 'learning',
 'love',
 'machine',
 'word2vec',
 'wordvec']

In [8]:
word2idx = {w:i for i,w in enumerate(vocab)}
# idx2word={i:w for w,i in word2idx.items()}
idx2word = {i:w for w,i in word2idx.items()}

In [9]:
idx2word

{0: 'algorithm',
 1: 'fun',
 2: 'great',
 3: 'i',
 4: 'implementing',
 5: 'is',
 6: 'learning',
 7: 'love',
 8: 'machine',
 9: 'word2vec',
 10: 'wordvec'}

In [10]:
vocab_size = len(vocab)

In [11]:
word2idx

{'algorithm': 0,
 'fun': 1,
 'great': 2,
 'i': 3,
 'implementing': 4,
 'is': 5,
 'learning': 6,
 'love': 7,
 'machine': 8,
 'word2vec': 9,
 'wordvec': 10}

In [12]:
X = []
Y = []
for sent in sentences:
    for i in range(len(sent)-1):
        X.append(word2idx[sent[i]])
        Y.append(word2idx[sent[i+1]])

In [13]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [14]:
print(X)
print(Y)

tensor([ 3,  7,  8, 10,  5,  2,  4,  9,  5])
tensor([7, 8, 6, 5, 2, 0, 9, 5, 1])


In [15]:
xx=X.reshape(3,3)

In [16]:
yy=Y.reshape(3,3)

In [17]:
vocab_size

11

In [18]:
class NextWordRNN(nn.Module):

    def __init__(self, vocab_size, embed_dim=10, hidden_dim=16):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, hx=self.rnn(x)
        out = out[:, -1, :]
        # out, _ = self.rnn(x)
        out = self.fc(out)
        return out

In [19]:
model = NextWordRNN(12000)

In [20]:
outputs=model(xx)

In [21]:
# inputs=X.unsqueeze(1)
# outputs=model(inputs)

In [22]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [23]:
for epoch in range(300):

    optimizer.zero_grad()

    inputs = X.unsqueeze(1)
    outputs = model(inputs)

    loss = loss_fn(outputs, Y)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 9.587372779846191
Epoch: 50 Loss: 0.5376880168914795
Epoch: 100 Loss: 0.21146628260612488
Epoch: 150 Loss: 0.18145087361335754
Epoch: 200 Loss: 0.17121200263500214
Epoch: 250 Loss: 0.1662171483039856


In [24]:
def predict_next(word):
    model.eval()
    idx = torch.tensor([[word2idx[word.lower()]]])

    with torch.no_grad():
        out = model(idx)
        pred = torch.argmax(out).item()

    return idx2word[pred]

In [25]:
print(predict_next("machine"))
print(predict_next("is"))
print(predict_next("wordvec"))

learning
great
is
